
# Analysis and generation of Sankey plots in EGFR mutant MADR scRNAseq datasets in contrast with human data #


The objectives of this Notebook are:\
1.- Comprehensive analysis of the cell types in the MADR-derived EGFR-dependent brain tumor models at single-cell level.\
2.- Similar analysis in human datasets recovered from (doi: 10.1038/s43018-022-00475-x)\
3.- Contrast the internal heterogeneity of MADR and human datasets of EGFR-dependent gliomas:  \
    3.1.- "Humanize" the murine datasets using an orthologous-gene human-mouse table\
    3.2.- Integration of Human and Mouse datasets in a single object\
    3.3.- Generate a Sankey (River) plot of the correlation between human and mouse cell types

In [ ]:
#Load libraries and set configurations
import numpy as np
from anndata import AnnData
import pandas as pd
import scanpy as sc
sc.settings.verbosity = 3 #verbosity: errors (0), warnings (1), info (2), hints (3)
#sc.settings.set_figure_params(dpi=600, dpi_save=1200, color_map='OrRd', figsize = (5,4))
#sc.settings.vector_friendly = False
sc.settings.n_jobs = 56
import scanpy.external as sce
import seaborn as sns
import loompy
import os, sys
import graphtools as gt
import scprep
#import cmocean
import sklearn
import scipy
import matplotlib.colors
import matplotlib.pyplot as plt
plt.rc('font', size=14)
import plotly.io as pio
pio.renderers.default='browser' #This set to plot the graph on the browser
#pio.renderers.default='svg' #This set to plot the graph in Spyder
import plotly.express as px
import plotly.graph_objects as go

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)


In [ ]:
# set the folder that you want as default for the results
os.chdir('/media/david/4TBNvMe/scRNAseq/Analysis/Adult Glioma')

## MADR-derived EGFR dependent tumor model analysis
The files we are using are already processed by Josh, so the analysis is a little bit different than usual. The different samples are already merged together and some of the grouping variables are already created such as sample_label which store the name of the sample from where each cell is from.

To do our analysis we will need to create some new grouping variables and start an "almost fresh" analysis on the merged object. We can call the raw file, which contains the normalized/logtransformed data and use that for our analysis. We wont need to filter bad quality cells, since Josh already did it but it is always good practice to check the QC plots

In [ ]:
# Load files
Mouse = sc.read('/media/david/4TBNvMe/scRNAseq/Analyzed datasets/EgfrviiiCleanMerge090822.h5ad')
# Remove all changes that Josh generated in the expression matrix by calling for the raw data
Mouse=Mouse.raw.to_adata()
# Add species and subtype grouping variables
Mouse.obs['species'] = 'Mouse'
new_names = {'Egfr glioma 1 hit': 'EgfrvIII',
             'Egfr glioma 3 hit Transplant Female':'EGFRvIII + Pten-Cdkn2a-Cas9',
             'Egfr glioma 3 hit Transplant Male':'EGFRvIII + Pten-Cdkn2a-Cas9',
             'Egfr glioma 3 hit female 1':'EGFRvIII + Pten-Cdkn2a-Cas9',
             'Egfr glioma 3 hit female 2':'EGFRvIII + Pten-Cdkn2a-Cas9',
             'Egfr glioma 3 hit male':'EGFRvIII + Pten-Cdkn2a-Cas9',
             'Egfr glioma 4 hit Nf1 organoid':'EGFRvIII + Pten-Cdkn2a-Nf1-Cas9',
             'Egfr glioma 4 hit Nf1 tissue':'EGFRvIII + Pten-Cdkn2a-Nf1-Cas9'}
Mouse.obs = Mouse.obs.assign(subtype = Mouse.obs.sample_label.map(new_names))

Mouse.layers['lognorm_counts'] = Mouse.X.copy() # stores in a layer the count matrix (.X) as is right now

Note: The 4 hits samples actually have Rb1 Cas9 too on them. We aren't keeping that in the label for simnplification

In [ ]:
#Create a copy to keep the original object unchanged to come back to it for following analysis
Mouse2 = Mouse.copy()

#Filter the object to keep only the samples that we are interested in
Mouse2_subset = Mouse2[~Mouse2.obs['sample_label'].isin(['Egfr glioma 4 hit Nf1 organoid', 
                                                         'Egfr glioma 3 hit Transplant Female',
                                                         'Egfr glioma 3 hit Transplant Male',
                                                         'Egfr glioma 3 hit female 2'])].copy()

#Perform basic filtering, calculate and plot some filtering parameters
sc.pp.filter_cells(Mouse2_subset, min_genes=200)
sc.pl.highest_expr_genes(Mouse2_subset, n_top=20, )

# store which genes are mitochondrial and which ones are ribosomal (just informative)
Mouse2_subset.var["mt"] = Mouse2_subset.var_names.str.startswith("mt-")
Mouse2_subset.var["ribo"] = Mouse2_subset.var_names.str.startswith(("Rps", "Rpl"))
# Calculate mitochondrial and ribosomal gene contributions as percentage of total counts
Mouse2_subset.obs['percent_mito'] = (np.sum(Mouse2_subset[:, Mouse2_subset.var['mt']].X, axis=1).A1/
                                     np.sum(Mouse2_subset.X, axis=1).A1)

Mouse2_subset.obs['percent_ribo'] = (np.sum(Mouse2_subset[:, Mouse2_subset.var['ribo']].X, axis=1).A1/
                                     np.sum(Mouse2_subset.X, axis=1).A1)

# n_genes and n_counts were previously calculated by Josh so we can just plot them without calculating it
sc.pl.violin(Mouse2_subset, ['n_genes', 'n_counts', 'percent_mito', 'percent_ribo'], jitter=0.4, multi_panel=True)
sc.pl.scatter(Mouse2_subset, x='n_counts', y='n_genes',color='sample_label')



The datasets were pre‑processed with good QC parameters, so no additional broad cell filtering is needed. However, after removing genes expressed in fewer than 3 cells, some cells may be left with no remaining expressed genes. These cells are not comparable to the rest of the dataset and should be removed (rows with sum = 0). As a precaution, we can also check for any all‑zero gene columns, though this should not occur given the initial gene filtering.

In [ ]:
print(np.any(Mouse2_subset.X.sum(axis=1) == 0))
print(np.any(Mouse2_subset.X.sum(axis=0) == 0))
Mouse2_subset = Mouse2_subset[:,Mouse2_subset.X.sum(axis=0).A1 > 0] 
print(np.any(Mouse2_subset.X.sum(axis=0) == 0))

Mouse2_subset.raw = Mouse2_subset # stores .X in the raw slot for some scanpy functions that use it later

Now we process the samples, Josh already calculated the PCA however, we have further filtered the dataset so the PCA calculated before maybe don't apply as well on the current cells. We also want to remove the effect of differences in sequencing deepth between cells by regressing out parameters related to them. for PCA analysis it is better to scale the data so the higher expressors don't dominate the PCA the batch effect by using harmony over the variable sample_label. After harmony we calculate the UMAP coordinates and do clustering using the leiden algorithm

In [ ]:
sc.pp.highly_variable_genes(Mouse2_subset, min_mean=0.0125, max_mean=6, min_disp=0.10)
sc.pl.highly_variable_genes(Mouse2_subset)

sc.pp.scale(Mouse2_subset)
Mouse2_subset.layers['scaled'] = Mouse2_subset.X.copy() # stores in a layer the count matrix (.X) as is right now

# Now we can calculate the PCA
sc.tl.pca(Mouse2_subset, svd_solver='arpack', n_comps=100)
sc.pl.pca_variance_ratio(Mouse2_subset, # This plot the variance explained by each PC,
                         log=True,      # it is useful to determine the relevant number
                         n_pcs = 100)   # of PCs to use in later functions such as sc.pp.neighbors()
Mouse2_subset.raw = Mouse2_subset
sc.external.pp.harmony_integrate(Mouse2_subset, key="sample_label", max_iter_harmony = 100)

In [ ]:
sc.pp.neighbors(Mouse2_subset, n_neighbors=50, n_pcs=30,use_rep="X_pca_harmony")
sc.tl.umap(Mouse2_subset)


After calculating the embedding we want to cluster our cells by their expressio profiles, I like to start with a highly granulated clustering that will split small clusters separate from similar bigger ones in case they are relevant by markers

In [ ]:
sc.tl.leiden(Mouse2_subset, resolution=3.5,flavor="igraph", n_iterations=2)


We can explore the clustering and expression of some general markers genes for some cell types:\
-postWPREV5gestalt = the MADR derived gene, it will mark MADR-transgenic cells\
-Gfap = Activated Astrocytes\
-Mog = Oligodendrocytes\
-Tubb3 = Neurons\
-Top2a = Cycling cells\
-Cldn5 = Endothelial cells\
-Pdgfrb = Pericytes\
-Foxj1 = Ependymal cells\
-Csf1r = Microglia\
-Xcr1 = Dendritic cells\
-Cd3g = T-Cells


In [ ]:
sc.pl.umap(Mouse2_subset, color=['leiden'], legend_fontsize = 5, size=20, legend_loc='on data', legend_fontoutline=1.5)
sc.pl.umap(Mouse2_subset,color=['postWPREV5gestalt','mTom','Top2a','Gfap','Mog','Tubb3',
                                'Cldn5','Pdgfrb','Foxj1','Csf1r','Xcr1','Cd3g'],
           color_map='magma_r', size=30, ncols=2,
           use_raw = False)#Remove doublets


This resolution is to granular so we can store it and run again the leiden clustering with a lower resolution

In [ ]:
#Mouse2_subset.obs['Leiden3.5'] = Mouse2_subset.obs.leiden
sc.tl.leiden(Mouse2_subset, resolution=1.5,flavor="igraph", n_iterations=2)
sc.pl.umap(Mouse2_subset, color=['Leiden3.5','leiden'], legend_fontsize = 8, legend_loc='on data', legend_fontoutline=2)

From the markers in the umap plots clusters 32 (Pdgfrb high, probably pericytes) in resolution 3.5 which is merged in resolution 1.5 shouldn't be merged, we can slice it out using the Leiden3.5 clustering in combination with the leiden clustering

Something similar happens with cluster 17 which is high in tubb3 a Neuron marker, and therefore it is probably a marker of Tumor Neuron-like cells.

In [ ]:
Mouse2_subset.obs['leiden'] = Mouse2_subset.obs['leiden'].cat.add_categories(['19', '20'])
Mouse2_subset.obs.loc[(Mouse2_subset.obs['Leiden3.5'] == '32') & (Mouse2_subset.obs['leiden'] == '15'), 'leiden'] = '19'
Mouse2_subset.obs.loc[(Mouse2_subset.obs['Leiden3.5'] == '17') & (Mouse2_subset.obs['leiden'] == '2'), 'leiden'] = '20'

sc.pl.umap(Mouse2_subset, color=['leiden'], legend_fontsize = 5, legend_loc='on data', legend_fontoutline=0.5)



In [ ]:
#sc.pp.normalize_total(Mouse2_subset, target_sum=1e4)
#sc.pp.log1p(Mouse2_subset)

sc.tl.rank_genes_groups(Mouse2_subset, groupby='leiden', layer='lognorm_counts', use_raw=False)
sc.pl.rank_genes_groups(Mouse2_subset, sharey=False)

In [ ]:
markers = sc.get.rank_genes_groups_df(Mouse2_subset, 
                                      group= None,
                                      pval_cutoff = 0.05,
                                      log2fc_min = 1)

columns = ['names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']
group_tables = []
group_labels = []  # will hold the group numbers for the top header row
unique_groups = markers['group'].unique()

for i, group in enumerate(unique_groups):
    group_df = markers[markers['group'] == group][columns].reset_index(drop=True)
    # Keep normal column names (no group prefix)
    group_tables.append(group_df)
    
    # Add spacer column except after the last group
    if i < len(unique_groups) - 1:
        spacer = pd.DataFrame(np.nan, index=group_df.index, columns=[""])
        group_tables.append(spacer)
    
    # For the top header row: group number only over 'names', blanks elsewhere
    group_labels.extend([group] + [""] * (len(columns) - 1))
    if i < len(unique_groups) - 1:
        group_labels.append("")  # spacer column has blank label

# Concatenate horizontally
final_table = pd.concat(group_tables, axis=1)

# Build MultiIndex for two header rows
final_table.columns = pd.MultiIndex.from_arrays([group_labels, final_table.columns])

# Save to CSV with multi-index headers
final_table.to_csv('Markers Mouse Leiden Clusters 2025_12_02.csv', index=False)
print('Saved: Markers Mouse Leiden Clusters 2025_12_02.csv')

Based in the markers we can now create a new clustering variable with the cell types that each cluster define

In [ ]:
#Simplified

new_names = {'0':'Tumor OPC-like',
             '1':'Tumor OPC-like',
             '2':'Tumor OPC-like', 
             '3':'Tumor OPC-like', 
             '4':'Tumor OPC-like', 
             '5':'Tumor Cycling', 
             '6':'Tumor OPC-like', 
             '7':'Tumor Cycling',
             '8':'Tumor Cycling',
             '9':'Tumor Oligo-like',
             '10':'Microglia', 
             '11':'Tumor Cycling',
             '12':'Endothelial Cells', 
             '13':'Tumor Cycling', 
             '14':'Oligodendrocytes', 
             '15':'Tumor Astro-like (mesenchymal-like)',
             '16':'Ependymal Cells', 
             '17':'Dendritic Cells', 
             '18':'Peripheral Macrophages',
             '19':'Pericytes',
             '20':'Tumor Neuron-like'}
Mouse2_subset.obs = Mouse2_subset.obs.assign(CellTypes_10x_Mouse = Mouse2_subset.obs.leiden.map(new_names))


In [ ]:
sc.pl.umap(Mouse2_subset, color=['leiden','CellTypes_10x_Mouse'], legend_fontsize = 5, legend_loc='on data', legend_fontoutline=1)


In [ ]:
# We can save a pdf file with these clusterings UMAPs
sc.pl.umap(Mouse2_subset, color=['CellTypes_10x_Mouse'],legend_fontsize = 7, legend_loc='on data', legend_fontoutline=1, 
           save=' - EGFRvIII 10X Mouse samples cell types named 2025_12_02.pdf')

# Additionally we can generate plots showing where the cells of each tumor subtype fall within the UMAP embedding
for subtype in ['EgfrvIII','EGFRvIII + Pten-Cdkn2a-Cas9','EGFRvIII + Pten-Cdkn2a-Nf1-Cas9']: 
    save_file = " - EGFRvIII 10X Mouse samples by subtypes - "+subtype+"2025_12_02.pdf"
    sc.pl.umap(Mouse2_subset, color='subtype', groups=[subtype], save=save_file, legend_loc=None)
    del save_file

Finally we can analyze the differential expression on the cell types defined before which have merge together some of the cluster determined by the Leiden algorithm. this differential expression genes should be clear markers for the cell types that we have named before

In [ ]:
# Generate markers and save them as a csv file
sc.tl.rank_genes_groups(Mouse2_subset, groupby='CellTypes_10x_Mouse', use_raw=False, layer='lognorm_counts')
sc.pl.rank_genes_groups(Mouse2_subset, sharey=False)

In [ ]:
markers = sc.get.rank_genes_groups_df(Mouse2_subset, 
                                      group= None,
                                      pval_cutoff = 0.05,
                                      log2fc_min = 1)
columns = ['names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']
group_tables = []
group_labels = []  # will hold the group numbers for the top header row
unique_groups = markers['group'].unique()

for i, group in enumerate(unique_groups):
    group_df = markers[markers['group'] == group][columns].reset_index(drop=True)
    # Keep normal column names (no group prefix)
    group_tables.append(group_df)
    
    # Add spacer column except after the last group
    if i < len(unique_groups) - 1:
        spacer = pd.DataFrame(np.nan, index=group_df.index, columns=[""])
        group_tables.append(spacer)
    
    # For the top header row: group number only over 'names', blanks elsewhere
    group_labels.extend([group] + [""] * (len(columns) - 1))
    if i < len(unique_groups) - 1:
        group_labels.append("")  # spacer column has blank label

# Concatenate horizontally
final_table = pd.concat(group_tables, axis=1)

# Build MultiIndex for two header rows
final_table.columns = pd.MultiIndex.from_arrays([group_labels, final_table.columns])

# Save to CSV with multi-index headers
final_table.to_csv('Markers 10X Mouse Cell Types 2025_12_02.csv', index=False)
print('Saved: Markers 10X Mouse Cell Types 2025_12_02.csv')


In [ ]:
#Cleanup objects

del [markers, new_names, subtype, final_table, group, group_df,group_labels, group_tables, i,columns, spacer, unique_groups]

In [ ]:
# Save the object
Mouse2_subset.write("/media/david/4TBNvMe/scRNAseq/Analyzed datasets/Mouse EgfrvIII tumors non-humanized with clusters named 2025_12_02.h5ad")

###Mouse Parse samples
We are going to process the Parse samples and check if we can combine them with the 10X

In [ ]:
# Define dataset paths in order
dataset_paths = [
    "/media/david/4TBNvMe/scRNAseq/Samples/Mouse/Katie Adult Tumor/Parse samples/Eviii-PCN-end-cntrl pool 1",
    "/media/david/4TBNvMe/scRNAseq/Samples/Mouse/Katie Adult Tumor/Parse samples/Eviii-PCN-end-cntrl pool 2",
    "/media/david/4TBNvMe/scRNAseq/Samples/Mouse/Katie Adult Tumor/Parse samples/Eviii-PC-endpoint-male",
    "/media/david/4TBNvMe/scRNAseq/Samples/Mouse/Katie Adult Tumor/Parse samples/Eviii-PC-endpoint-female"
]

# Collect AnnData objects in a list
adatas = []

for path in dataset_paths:
    adata = sc.read_mtx(f"{path}/count_matrix.mtx")
    gene_data = pd.read_csv(f"{path}/all_genes.csv")
    cell_meta = pd.read_csv(f"{path}/cell_metadata.csv")

    # Filter NaN genes
    gene_data = gene_data[gene_data.gene_name.notnull()]
    notNa = gene_data.index.to_list()
    adata = adata[:, notNa].copy()

    adata.var = gene_data
    adata.var.set_index('gene_name', inplace=True)
    adata.var.index.name = None
    adata.var_names_make_unique()

    adata.obs = cell_meta
    adata.obs.set_index('bc_wells', inplace=True)
    adata.obs.index.name = None
    adata.obs_names_make_unique()

    adatas.append(adata)

del [notNa, gene_data, path, cell_meta, dataset_paths, adata]

In [ ]:
[sc.pp.filter_cells(x, min_genes=200) for x in adatas]
list(map(sc.pp.scrublet, adatas))


In [ ]:
def plot_grid(plot_func, adatas, ncols=2, titles=None, figsize=(6, 4),
              hspace=0.4, wspace=0.3, **kwargs):
    """
    Arrange Scanpy plots in a grid.
    - Rasterizes sc.pl.violin(multi_panel=True) and sc.pl.scatter with color.
    - Embeds ax-based plots directly (e.g. sc.pl.highest_expr_genes).
    """
    n = len(adatas)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols,
                             figsize=(figsize[0]*ncols, figsize[1]*nrows))
    axes = axes.flatten()

    for i, adata in enumerate(adatas):
        ax = axes[i]

        # Rasterize if violin with multi_panel=True
        if plot_func is sc.pl.violin and kwargs.get("multi_panel", False):
            fg = plot_func(adata, show=False, **kwargs)
            fg.fig.canvas.draw()
            buf = np.frombuffer(fg.fig.canvas.buffer_rgba(), dtype=np.uint8)
            w, h = fg.fig.canvas.get_width_height()
            img = buf.reshape(h, w, 4)
            ax.imshow(img)
            ax.axis("off")
            plt.close(fg.fig)

        # Rasterize if scatter with color
        elif plot_func is sc.pl.scatter and "color" in kwargs:
            scatter_fig = plot_func(adata, show=False, **kwargs).figure
            scatter_fig.canvas.draw()
            buf = np.frombuffer(scatter_fig.canvas.buffer_rgba(), dtype=np.uint8)
            w, h = scatter_fig.canvas.get_width_height()
            img = buf.reshape(h, w, 4)
            ax.imshow(img)
            ax.axis("off")
            plt.close(scatter_fig)

        # Normal case: embed directly
        else:
            plot_func(adata, ax=ax, show=False, **kwargs)

        if titles:
            ax.set_title(titles[i])
        else:
            label = adata.obs['sample'].unique()[0]
            ax.set_title(str(label))


    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    plt.subplots_adjust(hspace=hspace, wspace=wspace)
    plt.show()

In [ ]:
plot_grid(sc.pl.highest_expr_genes, adatas, ncols=2, n_top=20)

In [ ]:
mito_genes = [x.var.index.str.startswith('mt-') for x in adatas]
percent_mito = [np.sum(x[:, x.var.index.str.startswith('mt-')].X, axis=1).A1/np.sum(x.X, axis=1).A1 for x in adatas]
for x,y in zip(adatas, percent_mito): x.obs['percent_mito'] = y
for x in adatas: x.obs['n_counts'] = x.X.sum(axis=1).A1
for x in adatas: x.obs['percent_ribo'] = np.sum(x[:, x.var_names.str.startswith(('Rps','Rpl'))].X, axis=1).A1 / np.sum(x.X, axis=1).A1

plot_grid(sc.pl.violin, adatas, ncols=1,
          keys=['n_genes', 'n_counts', 'percent_mito', 'percent_ribo', 'doublet_score'],
          jitter=0.4, multi_panel=True, figsize=(10,2))


In [ ]:
#thresholds for each sample based in their individual Violin plots
n_genes_up =    [10000,10000, 4000, 8000] 
n_genes_down =  [ 1000, 1000,  500, 1000]
n_counts_up =   [50000,60000,15000,40000]
mito_filter =   [  0.1,  0.1, 0.08, 0.04]
doublet_filter =[ 0.16, 0.14, 0.12, 0.16]

In [ ]:
adatas = [(a[a.obs.n_genes < b,:].copy()) for a,b in zip(adatas, n_genes_up)]
adatas = [(a[a.obs.n_genes > b,:].copy()) for a,b in zip(adatas, n_genes_down)]
adatas = [(a[a.obs.n_counts < b,:].copy()) for a,b in zip(adatas, n_counts_up)]
adatas = [(a[a.obs.percent_mito < b,:].copy()) for a,b in zip(adatas, mito_filter)]
adatas = [(a[a.obs.doublet_score < b,:].copy()) for a,b in zip(adatas, doublet_filter)]

plot_grid(sc.pl.violin,
          adatas,
          ncols=1,
          keys=['n_genes', 'n_counts', 'percent_mito', 'percent_ribo', 'doublet_score'],
          jitter=0.4,
          multi_panel=True,
         figsize=(10,2))

In [ ]:
plot_grid(sc.pl.scatter, adatas, ncols=2, figsize=(4,2.5), hspace=0.1, wspace=0.07,
          x='n_counts', y='n_genes', color='percent_mito')
plot_grid(sc.pl.scatter, adatas, ncols=2, figsize=(4,2.5), hspace=0.1, wspace=0.07,
          x='percent_mito', y='percent_ribo', color='n_genes')

[print('median transcipt count per cell: ' +  str(a.obs['sample'].unique()[0]) + ': ' +str(a.obs['n_counts'].median(0))) for a in adatas]
[print('median gene count per cell: '+  str(a.obs['sample'].unique()[0]) + ': '  + str(a.obs['n_genes'].median(0)))for a in adatas]
[print('median percent mito: '+  str(a.obs['sample'].unique()[0]) + ': '  + str(a.obs['percent_mito'].median(0))) for a in adatas]

for a in adatas: print(np.any(a.X.sum(axis=1) == 0))
for a in adatas: print(np.any(a.X.sum(axis=0) == 0))
adatas = [a[:,a.X.sum(axis=0) > 0].copy() for a in adatas]
for a in adatas: print(np.any(a.X.sum(axis=0) == 0))

In [ ]:
parse_mouse_original = sc.concat(adatas, join='outer', label="batch", merge= 'first')
parse_mouse_original.var_names_make_unique()
parse_mouse_original.obs_names_make_unique()
parse_mouse_original.obs['sample_label'] = parse_mouse_original.obs['sample'] 
del parse_mouse_original.obs['sample'] 

#del [doublet_filter, mito_filter, mito_genes, n_counts_up, n_genes_down, n_genes_up, percent_mito, x, y]

We can restore the .var and .uns slots to contain all the information in all adatas that has been lost in the merging

In [ ]:
# 1) Choose which var columns you want to restore/merge
cols_to_restore = ["gene_id", 'genome']  # extend as needed, e.g. ["gene_id", "gene_name"]

# 2) Build a DataFrame of restored columns from originals, aligned to the merged var index
restored = pd.DataFrame(index=parse_mouse_original.var.index)

for k, a in enumerate(adatas):
    # Align index types
    a.var.index = a.var.index.astype(str)
    # Optional: ensure same feature universe; skip missing features
    common_idx = restored.index.intersection(a.var.index)
    for col in cols_to_restore:
        if col in a.var.columns:
            restored.loc[common_idx, f"{col}-{k}"] = a.var.loc[common_idx, col]

# 3) Join restored columns back into the merged .var
parse_mouse_original.var = parse_mouse_original.var.join(restored, how="left")

# 4) Collapse suffixed columns into a single clean column per target
for col in cols_to_restore:
    cols = [c for c in parse_mouse_original.var.columns if c.startswith(f"{col}-")]
    if cols:
        # Prefer first non-null across datasets
        parse_mouse_original.var[col] = parse_mouse_original.var[cols].bfill(axis=1).iloc[:, 0]
        # If you prefer prioritizing dataset 0 values, use ffill instead:
        # adata.var[col] = adata.var[cols].ffill(axis=1).iloc[:, 0]
        parse_mouse_original.var.drop(columns=cols, inplace=True)

# 5) Collect scrublet results and  Merge into one dict keyed by batch
scrublets = [a.uns['scrublet'] for a in adatas if 'scrublet' in a.uns]
parse_mouse_original.uns['scrublet'] = {f"batch{i}": s for i, s in enumerate(scrublets)}

#Incompatible markers filtering
After cleanin up and setting our merged object and taking into account that it is from brain tumor samples we can take advantage of our markers and remove doublet that could have escape from the previous filters.

In [ ]:
adata = parse_mouse_original.copy()
#original = adata.copy()


#Filter incompatible markers
 #WPRE vs mTomato and mGFP
adata.X = np.asarray(adata.X.todense())
adata = adata[np.invert((adata[:,'postWPREV5gestalt'].X>0.2)& #Tumor
                        ((adata[:,'mTom'].X>0.2)|               #Non-Tumor
                         (adata[:,'mGFP'].X>0.2))), :]          #Cre transfected Non-Tumor

 #Tumor cells vs Microglia/Peripheral Macrophages
adata = adata[np.invert(((adata[:,'postWPREV5gestalt'].X>0.2)|  #Tumor
                         (adata[:,'Sox10'].X>0.2)|              #Tumor
                         (adata[:,'Pdgfra'].X>0.2)|             #Tumor
                         (adata[:,'Olig2'].X>0.2))&             #Tumor
                        ((adata[:,'Csf1r'].X>0.2)|              #Microglia
                         (adata[:,'C1qa'].X>0.2)|               #Microglia
                         (adata[:,'C1qb'].X>0.2)|               #Microglia
                         (adata[:,'Aif1'].X>0.2))), :]          #Microglia

 #T-Cells vs Tumor and Microglia/Peripheral Macrophages
adata = adata[np.invert((adata[:,'Cd3g'].X>0.2)&                #T-Cells
                        ((adata[:,'postWPREV5gestalt'].X>0.2)|  #Tumor
                         (adata[:,'Csf1r'].X>0.2))), :]         #Microglia

 #Pericytes vs Microglia/Peripheral Macrophages
adata = adata[np.invert((adata[:,'Pdgfrb'].X>0.2)&              #Pericytes
                        ((adata[:,'Csf1r'].X>0.2)|              #Microglia
                         (adata[:,'C1qa'].X>0.2)|               #Microglia
                         (adata[:,'C1qb'].X>0.2)|               #Microglia
                         (adata[:,'Aif1'].X>0.2))), :]          #Microglia

 #Endothelial cells vs Microglia/Peripheral Macrophages
adata = adata[np.invert((adata[:,'Cldn5'].X>0.2)&               #Endothelial cells
                        ((adata[:,'Csf1r'].X>0.2)|              #Microglia
                         (adata[:,'C1qa'].X>0.2)|               #Microglia
                         (adata[:,'C1qb'].X>0.2)|               #Microglia
                         (adata[:,'Aif1'].X>0.2))), :]          #Microglia

adata = parse_mouse_original[adata.obs.index,].copy()

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
sc.pl.highest_expr_genes(adata, n_top=20, )

#Normal processing
After removing doublets by double markers we can normally process the datasets

In [ ]:
adata.layers['counts']=adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
adata.layers['lognorm_counts']=adata.X.copy()
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=6, min_disp=0.10)
sc.pl.highly_variable_genes(adata)

#sc.pp.regress_out(adata, ['n_counts', 'percent_mito', 'percent_ribo', 'n_genes'], n_jobs=56)
sc.pp.scale(adata)

sc.tl.pca(adata, svd_solver='arpack', n_comps=100)
sc.pl.pca_variance_ratio(adata, log=True, n_pcs = 100)

In [ ]:
sc.external.pp.harmony_integrate(adata, key=['batch'],max_iter_harmony = 50)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=75,use_rep="X_pca_harmony")
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=3)

In [ ]:
sc.pl.umap(adata, color=['leiden'], legend_loc='on data', legend_fontsize=10, legend_fontoutline=0.7, add_outline=True)
sc.pl.umap(adata, color=['sample_label'], add_outline=True)
fig,ax = plt.subplots(nrows=2,ncols=2) 

for i,j in zip(adata.obs["batch"].unique().tolist(),ax.ravel()):
	sc.pl.umap(adata[adata.obs["batch"].isin([i])],legend_loc = None,color=["sample_label"],size=20,show=False,ax=j)

plt.tight_layout()
plt.show()

In [ ]:

sc.pl.umap(adata,color=['postWPREV5gestalt','mTom','mGFP',
                        'Pdgfra','Egfr','Top2a',
                        'Olig1', 'Olig2', 'Gfap',
                        'Cldn5','Pdgfrb', 'Kl',
                        'Foxj1','Csf1r','Itga4',
                        'Cd3g','Cd3e','Btla', 
                        'Vim', 'P2ry12', 'Ptprc' ],
           color_map='magma_r', size=20, ncols=3,add_outline=True)

In [ ]:
adata.obs['leiden3'] = adata.obs['leiden']
sc.tl.leiden(adata, resolution=2)

In [ ]:
del adata.uns['leiden_colors']
sc.pl.umap(adata, color=['leiden'], legend_loc='on data', legend_fontsize=10, legend_fontoutline=0.7, add_outline=True)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden')
sc.pl.rank_genes_groups(adata,sharey=False)

In [ ]:
markers = sc.get.rank_genes_groups_df(adata, 
                                      group= None,
                                      pval_cutoff = 0.05,
                                      log2fc_min = 1)
markers.to_csv('Markers Mouse parse samples Leiden Clusters 2025_12_02.csv')

#filter dataset based on Emily's previous analysis

In [ ]:
#Load Emily's metadata of the final object and use it to filter my adata object and add the cell type column
metadata = adata.obs
emily_metadata = pd.read_csv('/mnt/4TBNvMe/scRNAseq/Analysis/Adult Glioma/Emily final object metadata for Parse samples.csv')

merge_metadata = metadata.reset_index().merge(emily_metadata[['gene_count', 'tscp_count', 'mread_count','bc1_well', 'bc2_well', 'bc3_well','celltype']], on = ['gene_count', 'tscp_count', 'mread_count','bc1_well', 'bc2_well', 'bc3_well'], how='inner').set_index('index')

adata_filter = adata[merge_metadata.index].copy()
adata_filter.obs['Emily cell type'] = merge_metadata['celltype']

In [ ]:
#Let's plot the UMAP with Emily's cell type identifications

sc.pl.umap(adata_filter, color=['Emily cell type', 'leiden'], legend_loc='on data', legend_fontsize=7, legend_fontoutline=0.7, add_outline=True)


In [ ]:
sc.tl.rank_genes_groups(adata_filter, groupby='leiden')
sc.pl.rank_genes_groups(adata_filter,sharey=False)

markers = sc.get.rank_genes_groups_df(adata_filter, 
                                      group= None,
                                      pval_cutoff = 0.05,
                                      log2fc_min = 1)
markers.to_csv('Markers Mouse parse samples Leiden Clusters modified 031725-3.csv')

In [ ]:
#Rename the clusters
#Full naming
new_names = {'0':'Microglia','1':'Tumor OPC-like 3','2':'Microglia','3':'Tumor OPC-like 1',
             '4':'Tumor OPC-like 2','5':'Microglia','6':'Microglia','7':'Endothelial Cells',
             '8':'Tumor OPC-like 2','9':'Endothelial Cells','10':'Microglia','11':'Microglia Cycling',
             '12':'Tumor Cycling','13':'Microglia','14':'Pericytes','15':'Microglia',
             '16':'Tumor Astro-like','17':'Peripheral Macrophages','18':'Tumor OPC-like 1',
             '19':'Doublets Endothelial/Pericytes','20':'Tumor OPC-like 2','21':'Ependymal Cells',
             '22':'Endothelial Cells','23':'Microglia','24':'Doublets Tumor-Endothelial',
             '25':'Tumor Neuron-like','26':'Dendritic Cells','27':'Pericytes',
             '28':'Doublets Tumor-Microglia','29':'T Cells','30':'Choroid plexus','31':'Astrocytes',
             '32':'B Cells'}

adata_filter.obs = adata_filter.obs.assign(Cells = adata_filter.obs.leiden.map(new_names))


In [ ]:

sc.pl.umap(adata_filter, color=['Cells'], legend_loc='on data', legend_fontsize=6, legend_fontoutline=0.7, add_outline=True)
sc.pl.umap(adata_filter, color=['Emily cell type', 'Cells'], legend_loc='on data', legend_fontsize=6, legend_fontoutline=0.7, add_outline=True)

In [ ]:
#filter doublets clusters
adata_filter2 = adata_filter[~adata_filter.obs['Cells'].isin(['Doublets Endothelial/Pericytes',
                                                              'Doublets Tumor-Endothelial',
                                                              'Doublets Tumor-Microglia'])]


In [ ]:
sc.pl.umap(adata_filter2, color=['Cells'], legend_loc='on data', legend_fontsize=5, legend_fontoutline=1.5, add_outline=True,
           save=' - EGFRvIII Mouse parse samples cell types named 031725.pdf')

In [ ]:
def map_values(value):
    if value in ['Eviii-PC-endpoint-male', 'Eviii-PC-endpoint-female']:
        return 'EGFRvIII + Pten-Cdkn2a-Cas9'
    elif value == "Eviii-PCN-end-cntrl":
        return "EGFRvIII + Pten-Cdkn2a-Nf1-Cas9"
    else:
        return "Unknown"  # Optional: handle unexpected values

# Apply the function to create the new column
adata_filter2.obs['subtype'] = adata_filter2.obs['sample'].apply(map_values)

# Convert the new column to a categorical type
adata_filter2.obs['subtype'] = adata_filter2.obs['subtype'].astype('category')

# Verify the new column
print(adata_filter2.obs[['sample', 'subtype']].head())

In [ ]:

for subtype in ['EGFRvIII + Pten-Cdkn2a-Cas9', 'EGFRvIII + Pten-Cdkn2a-Nf1-Cas9']: 
    sc.pl.umap(adata_filter2, color='subtype', groups=[subtype], legend_loc=None)

In [ ]:

del [ax,  emily_metadata, fig, i, j, markers, merge_metadata, metadata, new_names, subtype]

In [ ]:
adata_filter2.write('Mouse Parse EgfrvIII tumors non-humanized with clusters named 031725.h5ad')

### Integrate 10X and Parse Mouse datasets

In [ ]:
# Create a backup of both 10X and parse datasets
Mouse_10X_backup = Mouse.copy()
Mouse_parse_backup = parse_mouse_original.copy()

In [ ]:
#filter cells using the processed object
Mouse_10X_clean = Mouse_10X_backup[Mouse2_subset.obs.index].copy()
Mouse_10X_clean.obs['tech'] = '10X'
Mouse_10X_clean.obs['cells'] = Mouse2_subset.obs['Leiden3']

Mouse_Parse_clean = Mouse_parse_backup[adata_filter2.obs.index].copy()
Mouse_Parse_clean.obs['tech'] = 'Parse'
Mouse_Parse_clean.obs['subtype'] = adata_filter2.obs['subtype']
Mouse_Parse_clean.obs['cells'] = adata_filter2.obs['Cells']
Mouse_Parse_clean.obs['sample_label'] = Mouse_Parse_clean.obs['sample']

In [ ]:
adata_mouse = Mouse_10X_clean.concatenate(Mouse_Parse_clean,join='inner')

In [ ]:
sc.pp.filter_cells(adata_mouse, min_genes=200)
sc.pp.filter_genes(adata_mouse, min_cells=3)
sc.pl.highest_expr_genes(adata_mouse, n_top=20, )

sc.settings.set_figure_params(dpi=300, dpi_save=1200, color_map='OrRd', figsize = (6,6))

adata_mouse.obs['percent_mito'] = np.sum(adata_mouse[:, adata_mouse.var.index.str.startswith('mt-')].X, axis=1).A1 / np.sum(adata_mouse.X, axis=1).A1
adata_mouse.obs['n_counts'] = adata_mouse.X.sum(axis=1).A1
adata_mouse.obs['percent_ribo'] = np.sum(adata_mouse[:, adata_mouse.var_names.str.startswith(('Rps','Rpl'))].X, axis=1).A1 / np.sum(adata_mouse.X, axis=1).A1

sc.pl.violin(adata_mouse, ['n_genes', 'n_counts', 'percent_mito', 'percent_ribo'], jitter=0.4, multi_panel=True)

In [ ]:
adata_mouse.raw = adata_mouse
sc.pp.normalize_total(adata_mouse, target_sum=1e4)
sc.pp.log1p(adata_mouse)
sc.pp.highly_variable_genes(adata_mouse, min_mean=0.0125, max_mean=6, min_disp=0.10)
sc.pl.highly_variable_genes(adata_mouse)

In [ ]:

sc.pp.regress_out(adata_mouse, ['n_counts', 'percent_mito', 'percent_ribo', 'n_genes'], n_jobs=56)
sc.pp.scale(adata_mouse)
sc.tl.pca(adata_mouse, svd_solver='arpack', n_comps=100)
sc.pl.pca_variance_ratio(adata_mouse, log=True, n_pcs = 100)

In [ ]:
sc.external.pp.harmony_integrate(adata_mouse, key=['sample_label', 'tech'],
                                 max_iter_harmony = 100,
                                 #lamb=0.8, sigma=0.05
                                )

In [ ]:
sc.pp.neighbors(adata_mouse, n_neighbors=20, n_pcs=25,knn=True,use_rep="X_pca_harmony")
sc.tl.umap(adata_mouse)


In [ ]:
sc.tl.leiden(adata_mouse, resolution=2)

In [ ]:
sc.settings.set_figure_params(figsize = (10, 8))

sc.pl.umap(adata_mouse, color=['leiden', 'cells','tech'], ncols=1,size=10, add_outline =True, legend_loc='on data', legend_fontoutline=1.5, legend_fontsize=15)


sc.settings.set_figure_params(figsize = (6,3))

fig,ax = plt.subplots(nrows=1,ncols=2)  
for i,j in zip(adata_mouse.obs["tech"].unique().tolist(),ax.ravel()):
	sc.pl.umap(adata_mouse[adata_mouse.obs["tech"].isin([i])], color=["tech"],size=10, legend_loc = None, show=False,ax=j)
plt.tight_layout()
plt.show()


In [ ]:

sc.settings.set_figure_params(figsize = (4,4))

sc.pl.umap(adata_mouse,color=['postWPREV5gestalt','mTom','mGFP',
                        'Pdgfra','Egfr','Top2a',
                        'Olig1', 'Olig2', 'Gfap',
                        'Cldn5','Pdgfrb', 'Kl',
                        'Foxj1','Csf1r','Itga4',
                        'Cd3g','Cd3e','Btla', 
                        'Vim', 'P2ry12', 'Ptprc' ],
           color_map='magma_r', size=20, ncols=3,add_outline=True)

Tumor cells are falling on Microglia clusters which are probably doublets. We can remove those cells

In [ ]:

adata_mouse_filtered = adata_mouse[~((adata_mouse.raw.to_adata()[:, 'postWPREV5gestalt'].X.toarray().flatten() > 0) & adata_mouse.obs['leiden'].isin(['10', '17', '8', '1', '12', '5', '16'])), :].copy()

In [ ]:
#adata_mouse_filtered.obs['Leiden2'] = adata_mouse_filtered.obs['leiden']
sc.settings.set_figure_params(figsize = (6,5))
sc.tl.leiden(adata_mouse_filtered, resolution=8)
sc.pl.umap(adata_mouse_filtered, color=['leiden', 'cells'], size=10, add_outline =True, legend_loc='on data', legend_fontoutline=1.5)


In [ ]:
#adata_mouse_filtered.obs['Leiden2'] = adata_mouse_filtered.obs['Leiden2'].cat.add_categories(['22','23','24'])
adata_mouse_filtered.obs.loc[(adata_mouse_filtered.obs['Leiden2'] == '11') & (adata_mouse_filtered.obs['leiden'] == '74'), 'Leiden2'] = '22'
adata_mouse_filtered.obs.loc[(adata_mouse_filtered.obs['Leiden2'] == '9') & (adata_mouse_filtered.obs['leiden'] == '85'), 'Leiden2'] = '23'
adata_mouse_filtered.obs.loc[(adata_mouse_filtered.obs['Leiden2'] == '12') & (adata_mouse_filtered.obs['leiden'] == '92'), 'Leiden2'] = '24'



In [ ]:

sc.pl.umap(adata_mouse_filtered, color=['Leiden2'], size=10, add_outline =True, legend_loc='on data', legend_fontoutline=1.5)


In [ ]:
sc.settings.set_figure_params(figsize = (15,5))

fig,ax = plt.subplots(nrows=1,ncols=2)  
for i,j in zip(adata_mouse_filtered.obs["tech"].unique().tolist(),ax.ravel()):
	sc.pl.umap(adata_mouse_filtered[adata_mouse_filtered.obs["tech"].isin([i])], title=i,color=["cells"],size=10, show=False,ax=j)
plt.tight_layout()
plt.show()


In [ ]:
adata_mouse_filtered.obs.leiden =adata_mouse_filtered.obs['Leiden2']
#del adata_mouse.obs['Leiden2']

In [ ]:

sc.settings.set_figure_params(figsize = (4,4))

sc.tl.rank_genes_groups(adata_mouse_filtered, groupby='leiden')
sc.pl.rank_genes_groups(adata_mouse_filtered,sharey=False)

In [ ]:
markers = sc.get.rank_genes_groups_df(adata_mouse_filtered, 
                                      group= None,
                                      pval_cutoff = 0.05,
                                      log2fc_min = 1)

grouped = markers.groupby('group', observed=True)

dfs = []

for group_name, group_df in grouped:
    # Drop the grouping column (A) if you don't need it in each part of the final dataframe
    group_df = group_df.drop(columns='group')

    # Reset index to ensure proper alignment when concatenating
    group_df.reset_index(drop=True, inplace=True)

    # Rename the columns to include the group name (e.g., sample 1_B, sample 1_C)
    group_df.columns = [f"{group_name}_{col}" for col in group_df.columns]

    # Append the transformed group to the list
    dfs.append(group_df)
    
result = pd.concat(dfs, axis=1)

result.to_csv('Markers Mouse parse&10X samples Leiden Clusters 031725.csv')

In [ ]:

new_names = {'0':'Endothelial Cells','1':'Microglia','2':'Tumor OPC-like 3',
             '3':'Tumor OPC-like 2','4':'Tumor OPC-like 1','5':'Microglia',
             '6':'Tumor OPC-like 2','7':'Tumor Cycling','8':'Microglia',
             '9':'Tumor Neuron-like','10':'Microglia','11':'Tumor Astro-like',
             '12':'Microglia','13':'Tumor Neuron-like','14':'Tumor OPC-like 2',
             '15':'Pericytes','16':'Peripheral Macrophages','17':'Microglia Cycling',
             '18':'Peripheral Immune','19':'Oligodendrocytes','20':'Ependymal Cells',
             '21':'Pericytes','22':'Astrocytes','23':'Tumor Neuron-like','24':'Microglia'}
adata_mouse_filtered.obs = adata_mouse_filtered.obs.assign(CellType = adata_mouse_filtered.obs.leiden.map(new_names))

new_names = {'0':'Endothelial Cells','1':'Microglia','2':'Tumor-OPC-like',
             '3':'Tumor OPC-like','4':'Tumor OPC-like','5':'Microglia',
             '6':'Tumor OPC-like','7':'Tumor Cycling','8':'Microglia',
             '9':'Tumor Neuron-like','10':'Microglia','11':'Tumor Astro-like',
             '12':'Microglia','13':'Tumor Neuron-like','14':'Tumor OPC-like',
             '15':'Pericytes','16':'Peripheral Macrophages','17':'Microglia Cycling',
             '18':'Peripheral Immune','19':'Oligodendrocytes','20':'Ependymal Cells',
             '21':'Pericytes','22':'Astrocytes','23':'Tumor Neuron-like','24':'Microglia'}
adata_mouse_filtered.obs = adata_mouse_filtered.obs.assign(CellType_Simplified = adata_mouse_filtered.obs.leiden.map(new_names))

In [ ]:

adata_mouse_filtered.uns['CellType_colors'] = ['#4F81BD', #'Astrocytes',
                                               '#43A047', #'Endothelial Cells',
                                               '#81C784', #'Ependymal Cells',
                                               '#984EA3', #'Microglia',
                                               '#b29cdb', #'Microglia Cycling',
                                               '#A6CEE3', #'Oligodendrocytes',
                                               '#66BB6A', #'Pericytes',
                                               '#CE93D8', #'Peripheral Immune',
                                               '#E1BEE7', #'Peripheral Macrophages'
                                               '#FF7043', #'Tumor Astro-like',
                                               '#FF5252', #'Tumor cycling',
                                               '#C62828', #'Tumor Neuron-like',
                                               '#c49c94', #'Tumor OPC-like 1',
                                               '#f7b6d2', #'Tumor OPC-like 2',
                                               '#9edae5'#'Tumor OPC-like 3',
                                              ]
                                               

In [ ]:

sc.settings.set_figure_params(figsize = (10,8))

sc.pl.umap(adata_mouse_filtered, color=['CellType'], size=10, add_outline =True, 
           legend_loc='on data', legend_fontoutline=1.5, legend_fontsize = 10,
          save = ' - Final Cell named clustering integration mouse 10X and Parse.pdf')

sc.pl.umap(adata_mouse_filtered, color=['CellType_Simplified'], size=10, add_outline =True, 
           legend_loc='on data', legend_fontoutline=1.5, legend_fontsize = 10,
          save = ' - Final Cell named simplified microglia clustering integration mouse 10X and Parse.pdf')

In [ ]:
for subtype in adata_mouse_filtered.obs.subtype.unique().tolist(): 
    save_file = " - EGFRvIII Mouse parse&10X samples by subtypes - "+subtype+" 031725.pdf"
    sc.pl.umap(adata_mouse_filtered, color='subtype', groups=[subtype], size=30, save=save_file, legend_loc=None)
    del save_file

In [ ]:
adata_mouse_filtered.write('Final EgfrvIII mouse tumor 10X and parse integration datasets 031725.h5ad')

In [ ]:
cnv.io.genomic_position_from_biomart(adata_mouse_filtered, species='mmusculus',inplace=True,biomart_gene_id='mgi_symbol')
cnv.tl.infercnv(
    adata_mouse_filtered,
    reference_key="CellType_Simplified",
    reference_cat=['Astrocytes', 'Endothelial Cells', 'Ependymal Cells', 
                   'Microglia','Microglia Cycling', 'Peripheral Macrophages', 
                   'Peripheral Immune', 'Pericytes','Oligodendrocytes'], # non-tumor references
    layer = 'lognorm')
cnv.tl.cnv_score(adata_mouse_filtered, groupby = 'CellType_Simplified')

In [ ]:
#Subsample the addata object so the clusters are better represented on the heatmap
group_key = "CellType_Simplified"
max_cells_per_cluster = 100  # adjust as needed

# Subsample each cluster
subsampled_indices = (
    adata_mouse_filtered.obs
    .groupby(group_key)
    .apply(lambda x: x.sample(min(len(x), max_cells_per_cluster), random_state=42))
    .index.get_level_values(1)
)

# Create a new AnnData object with subsampled cells
adata_sub = adata_mouse_filtered[subsampled_indices].copy()

# Build chromosome labels from chr_pos
chromosomes = list(adata_sub.uns['cnv']['chr_pos'].keys())
adata_sub.uns['cnv']['var_group_labels'] = chromosomes

In [ ]:
sns.set_context("talk", font_scale=0.7)  # "paper", "notebook", "talk", "poster"
cnv.pl.chromosome_heatmap(adata_sub,
                          groupby='CellType_Simplified',
                          cmap="seismic",       # diverging colormap
                          figsize=(9, 18),     # compress vertically
                          dendrogram=True,     # optional: remove dendrogram for clarity
                          vmin=-0.15, vmax= 0.15, 
                         )

In [ ]:
marker_panel = {
    "Peripheral Immune (T Cells)": [
        "Cd3d", "Cd2", "Cd3e", "Cd8a"
    ],
    "Peripheral Macrophages": [
        "Msr1", "Siglec1", "Ccr2", "Cd68",
    ],
    "Microglia": [
        "Tmem119", "P2ry12", "Trem2", "Sall1"
    ],
    "Cycling": [
        "Mki67", "Top2a", "Cenpf", "Pcna", "Cenpe"
    ],
    "Astrocytes / Tumor Astro-like": [
        "Gfap", "Aqp4", "Aldh1l1", "Slc1a3", "Glul"
    ],
    "Tumor Neuron-like": [
        "Map2", "Gria1", "Gria2", "Gria3", "Scn8a", "Nrxn1", "Nrxn3"
    ],
    "Tumor OPC-like": [
        "Pdgfra", "Sox6", "Sox9", "Nkx2-2"
    ],
    "Endothelial Cells": [
        'Cldn5', "Pecam1", "Cdh5"
    ],
    "Pericytes": [
        "Pdgfrb", "Rgs5", "Des"
    ],
    "Mature Oligodendrocytes": [
        "Mbp", "Mog", "Plp1"
    ],
    "Ependymal Cells": [
        "Foxj1", "Dnah11", "Cfap44"
    ]
}


In [ ]:
sc.pl.dotplot(
    adata_mouse_filtered,
    marker_panel,
    groupby="CellType_Simplified",
        standard_scale="var",
    dendrogram=True,
    save = '2025_12_09 - Mouse EGFR final object dotplot cell types markers.pdf'
    )

## Human EGFR dependent tumor datasets analysis
The datasets used for this analysis come from and atlas of human brain tumors (doi: 10.1038/s43018-022-00475-x). 

As with the mouse datasets we need to add the grouping variable species which will be necessary for the future cross-species integration.Additionally, the atlas contain several different types of tumors that we are not interested in analyzing, so we must filter the object to collect only the interesting subtypes

I have preprocess in a separate script this atlas to remove bad quality cells in the individual datasets. Therefore, the QC in the next steps will not remove too many cells.

In [ ]:
Human = sc.read('/media/david/4TBNvMe/scRNAseq/Analysis/Adult Glioma/Human EGFR datasets complete raw.h5ad')
Human.obs['species'] = 'Human'
Human_immune = Human[Human.obs["subtype"].isin (['EGFRsnv','EGFRamp + NF1', 'EGFRamp, EGFRsnv, PTEN & CDKN2A'])].copy()



sc.pp.filter_cells(Human_immune, min_genes=200)
sc.pp.filter_genes(Human_immune, min_cells=3)
sc.pl.highest_expr_genes(Human_immune, n_top=20, )

Human_immune.obs['percent_mito'] = np.sum(Human_immune[:, Human_immune.var.index.str.startswith('MT-')].X, axis=1).A1 / np.sum(Human_immune.X, axis=1).A1
Human_immune.obs['n_counts'] = Human_immune.X.sum(axis=1).A1
Human_immune.obs['percent_ribo'] = np.sum(Human_immune[:, Human_immune.var_names.str.startswith('RPS','RPL')].X, axis=1).A1 / np.sum(Human_immune.X, axis=1).A1
sc.pl.violin(Human_immune, ['n_genes', 'n_counts', 'percent_mito', 'percent_ribo'], jitter=0.4, multi_panel=True)

In [ ]:
print(np.any(Human_immune.X.sum(axis=0) == 0))
print(np.any(Human_immune.X.sum(axis=1) == 0))
Human_immune = Human_immune[:,Human_immune.X.sum(axis=0) > 0]
print(np.any(Human_immune.X.sum(axis=0) == 0))

We want to add another sample that it is NF1. This dataset is composed of 4 samples all from the same patient. 2 of them are primary tumors and 2 are recurrence, probably from different region/pieces of tumor

In [ ]:
Human_NF1 = sc.read('/media/david/4TBNvMe/scRNAseq/Samples/Human/Human samples EGFR gliomas/04.-NF1/index_patient_merged.h5ad')

The different samples are separated by a prefix on their barcode, but we need it to be a column that can be used for batch correction

In [ ]:
prefix_map = {
    'P1': 'NF1 Primary 1',
    'P2': 'NF1 Primary 2',
    'R1': 'NF1 Recurrence 1',
    'R2': 'NF1 Recurrence 2'
}

# Extract prefix from each index and map to classification
Human_NF1.obs['sample_label'] = Human_NF1.obs_names.to_series().apply(
    lambda x: next((label for prefix, label in prefix_map.items() if x.startswith(prefix)), 'Unknown')
)

Human_NF1.obs['subtype'] = 'EGFRamp + NF1'

In [ ]:

sc.pp.filter_cells(Human_NF1, min_genes=200)
sc.pp.filter_genes(Human_NF1, min_cells=3)
sc.pl.highest_expr_genes(Human_NF1, n_top=20, )

Human_NF1.obs['percent_mito'] = np.sum(Human_NF1[:, Human_NF1.var.index.str.startswith('MT-')].X, axis=1).A1 / np.sum(Human_NF1.X, axis=1).A1
Human_NF1.obs['n_counts'] = Human_NF1.X.sum(axis=1).A1
Human_NF1.obs['percent_ribo'] = np.sum(Human_NF1[:, Human_NF1.var_names.str.startswith('RPS','RPL')].X, axis=1).A1 / np.sum(Human_NF1.X, axis=1).A1

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))  # 2x2 grid
axes = axes.flatten()

metrics = ['n_genes', 'n_counts', 'percent_mito', 'percent_ribo']

for ax, metric in zip(axes, metrics):
    sc.pl.violin(
        Human_NF1,
        metric,
        groupby='sample_label',
        jitter=0.4,
        multi_panel=False,   # single panel per call
        ax=ax,
        show=False           # don’t auto-show, plot into given axis
    )

plt.tight_layout()
plt.show()

In [ ]:
# thresholds per sample
thresholds = {
    'NF1 Primary 1': {'min_genes': 200, 'max_genes': 6000, 'max_counts': 30000, 'max_mito': 0.02},
    'NF1 Primary 2': {'min_genes': 300, 'max_genes': 6000, 'max_counts': 30000, 'max_mito': 0.02},
    'NF1 Recurrence 1': {'min_genes': 250, 'max_genes': 7000, 'max_counts': 30000, 'max_mito': 0.02},
    'NF1 Recurrence 2': {'min_genes': 200, 'max_genes': 7000, 'max_counts': 30000, 'max_mito': 0.02}
}


In [ ]:
filtered_subsets = []

for sample, th in thresholds.items():
    # Subset AnnData for this sample
    adata_sample = Human_NF1[Human_NF1.obs['sample_label'] == sample].copy()
    
    # Apply filters
    adata_sample = adata_sample[
        (adata_sample.obs['n_genes'] >= th['min_genes']) &
        (adata_sample.obs['n_genes'] <= th['max_genes']) &
        (adata_sample.obs['n_counts'] <= th['max_counts']) &
        (adata_sample.obs['percent_mito'] <= th['max_mito'])
    ].copy()
    
    filtered_subsets.append(adata_sample)

# Concatenate back into one AnnData
Human_NF1_filtered = filtered_subsets[0].concatenate(*filtered_subsets[1:], batch_key="sample_label")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))  # 2x2 grid
axes = axes.flatten()

metrics = ['n_genes', 'n_counts', 'percent_mito', 'percent_ribo']

for ax, metric in zip(axes, metrics):
    sc.pl.violin(
        Human_NF1_filtered,
        metric,
        groupby='sample_label',
        jitter=0.4,
        multi_panel=False,   # single panel per call
        ax=ax,
        show=False           # don’t auto-show, plot into given axis
    )

plt.tight_layout()
plt.show()

Now we can merge the NF1 datasets with the other Human datasets

In [ ]:
Human_immune = Human_immune.concatenate(Human_NF1_filtered, join='outer')

Human_backup = Human_immune.copy()

Again we want to save the raw data after normalization and log transformation. We can also check the variable genes

In [ ]:

sc.pp.normalize_total(Human_immune, target_sum=1e4)
sc.pp.log1p(Human_immune)
Human_immune.raw = Human_immune

sc.pp.highly_variable_genes(Human_immune, min_mean=0.0125, max_mean=6, min_disp=0.10)
sc.pl.highly_variable_genes(Human_immune)

Next we do the standard precessing where we regress out the effect of the following variables:
-ncounts, ngenes : It is important to take into account that a cell that have been sequencer at a deeper level could present and artefactual difference in the expression of genes which are lowly expressed
percent_mito, percent_ribo: Differences in the mitocondrial and ribosomal genes fraction of the total genes between the cells can generate artefacts that can be removed

NOTE: while regressing out the effect of ncounts and ngenes is a completely standard procedure there are different opinions about the regressing out by the mitochondrial and ribosomal content, however in my experience it doesn't make extreme changes in the results and the changes usually are for the better. In any case is something to take in to account when analyzing other datasets.

In [ ]:

sc.pp.regress_out(Human_immune, ['n_counts', 'percent_mito', 'percent_ribo', 'n_genes'], n_jobs=56)

After regressing_out the expression matrix we can proceed to scale and calculate PCA. Next we perform batch correction with harmony and calculate the UMAP embedding and Leiden clustering.

Note: harmony batch correction is usually neccesary to obtain a good integration between different datasets, however you can skip this step to check how the umap embedding looks like without batch correction in case you believe there could be some overcorrection in the harmony results.

In [ ]:
sc.pp.scale(Human_immune)
sc.tl.pca(Human_immune, svd_solver='arpack', n_comps=100)
sc.pl.pca_variance_ratio(Human_immune, log=True, n_pcs = 100)

In [ ]:
sc.external.pp.harmony_integrate(Human_immune, key=['sample_label'],max_iter_harmony = 50)
sc.pp.neighbors(Human_immune, n_neighbors=100, n_pcs=100,use_rep="X_pca_harmony")
sc.tl.umap(Human_immune, maxiter=100)
sc.tl.leiden(Human_immune, resolution=1.3)

We can now plot the results of the UMAP embedding and clustering

In [ ]:
sc.pl.umap(Human_immune, color=['leiden'], legend_loc='on data', legend_fontsize=15,legend_fontoutline=0.5)

# To make the subtype comparable to those in the mouse datasets I am personalizing the colors used for the subtype variable
Human_immune.uns['subtype_colors'] = ['#fd9f4b','#5ba1d3', '#1fe98b']
sc.pl.umap(Human_immune, color=['subtype'], )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))  # adjust grid size to your number of subtypes
axes = axes.flatten()

subtype = Human_immune.obs['subtype'].unique()

for ax, subtypes in zip(axes, subtype):
    sc.pl.umap(Human_immune[Human_immune.obs['subtype'] == subtypes].copy(),
               color='leiden', title=f"UMAP - {subtypes}", ax=ax, show=False)

plt.tight_layout()
plt.show()

And the markers for diferent cell types

In [ ]:

sc.pl.umap(Human_immune,color=['GFAP','MOG','TOP2A','CSF1R','CLDN5','PDGFRB','CD3G','FOXJ1', 'SOX11'],color_map='magma_r', size=5, ncols=3)


As with the mouse data we want to do the differential expression analysis in the leiden clustering to be able to determine which cells are represented by each cluster

In [ ]:
sc.tl.rank_genes_groups(Human_immune, groupby='leiden')
sc.pl.rank_genes_groups(Human_immune, sharey=False)

In [ ]:
markers = sc.get.rank_genes_groups_df(Human_immune, 
                                      group= None,
                                      pval_cutoff = 0.05,
                                      log2fc_min = 1)

columns = ['names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']
group_tables = []
group_labels = []  # will hold the group numbers for the top header row
unique_groups = markers['group'].unique()

for i, group in enumerate(unique_groups):
    group_df = markers[markers['group'] == group][columns].reset_index(drop=True)
    # Keep normal column names (no group prefix)
    group_tables.append(group_df)
    
    # Add spacer column except after the last group
    if i < len(unique_groups) - 1:
        spacer = pd.DataFrame(np.nan, index=group_df.index, columns=[""])
        group_tables.append(spacer)
    
    # For the top header row: group number only over 'names', blanks elsewhere
    group_labels.extend([group] + [""] * (len(columns) - 1))
    if i < len(unique_groups) - 1:
        group_labels.append("")  # spacer column has blank label

# Concatenate horizontally
final_table = pd.concat(group_tables, axis=1)

# Build MultiIndex for two header rows
final_table.columns = pd.MultiIndex.from_arrays([group_labels, final_table.columns])

# Save to CSV with multi-index headers
final_table.to_csv("Markers_Human_Clustering_leiden_111125.csv", index=False)
print("Saved: Markers_Human_Clustering_leiden_111125.csv")


We can name the clusters after reviewing their markers

In [ ]:
new_names = {'0':'Tumor - OPC-like 1',
             '1':'Microglia',
             '2':'Oligodendrocytes',
             '3':'Tumor - OPC-like 2',
             '4':'Macrophages',
             '5':'Tumor - OPC-like 3',
             '6':'Tumor - Mesenchymal-like',
             '7':'Tumor - Neuron-like',
             '8':'Tumor - Cycling',
             '9':'Doublets',
             '10':'Tumor - Astro-like',
             '11':'T Cells',
             '12':'Dendritic Cells',
             '13':'Neurons',
             '14':'Doublets/dying cells',
             '15':'Monocytes with dendritic-like features',
             '16':'Tumor - OPC-like 4',
             '17':'Oligodendrocytes',
             '18':'Endothelial Cells',
             '19':'Tumor - Astro-like 2',
             '20':'Macrophages',
             '21':'Astrocytes',
             '22':'Tumor - Astro-like',
             '23':'Pericytes',
             '24':'Oligodendrocytes',
             '25':'Tumor - OPC-like 5',
             '26':'Tumor - Neuron-like 2'}

Human_immune.obs = Human_immune.obs.assign(CellType1 = Human_immune.obs.leiden.map(new_names))
sc.pl.umap(Human_immune,color=['CellType1'], legend_loc = 'on data', legend_fontsize=5, add_outline=True)

After analyzing the clusters-gene markers the are clusters that need to be removed since they look like dying cells or doublets. After donig the filtering we will need to start again from the raw data with the filtered datasets, but we can save the information of this processing in the new dataset so we can see how it changes.

In [ ]:
# filter doublet clusters
Human_immune_filter = Human_immune[~Human_immune.obs['CellType1'].isin(['Doublets', 'Doublets/dying cells'])]

In [ ]:
# restart the datasets using the backup merged version
mask = Human_backup.obs_names.isin(Human_immune_filter.obs_names)
Human_immune = Human_backup[mask].copy()

del mask

In [ ]:
#Keep the CellType identity and UMAP coordinates
Human_immune.obs['Original CellType']= Human_immune_filter.obs['CellType1']
Human_immune.obsm['Original X_umap'] = Human_immune_filter.obsm['X_umap']
del Human_immune.var


In [ ]:
#Reprocess the dataset
Human_immune.layers['counts'] = Human_immune.X.copy()

sc.pp.normalize_total(Human_immune, target_sum=1e4)
sc.pp.log1p(Human_immune)
Human_immune.layers['lognorm'] = Human_immune.X.copy()
Human_immune.raw = Human_immune

sc.pp.highly_variable_genes(Human_immune, min_mean=0.0125, max_mean=6, min_disp=0.10)
sc.pl.highly_variable_genes(Human_immune)

In [ ]:
sc.pp.regress_out(Human_immune, ['n_counts', 'percent_mito', 'percent_ribo', 'n_genes'], n_jobs=56)

In [ ]:
sc.pp.scale(Human_immune)
Human_immune.layers['scaled'] = Human_immune.X.copy()


sc.tl.pca(Human_immune, svd_solver='arpack', n_comps=100)
sc.pl.pca_variance_ratio(Human_immune, log=True, n_pcs = 100)

In [ ]:

sc.external.pp.harmony_integrate(Human_immune, key=['sample_label'],max_iter_harmony = 50)

In [ ]:
sc.pp.neighbors(Human_immune, n_neighbors=200, n_pcs=100,use_rep="X_pca_harmony")
sc.tl.umap(Human_immune, maxiter=100)
sc.tl.leiden(Human_immune, resolution=1.3)

In [ ]:
sc.pl.umap(Human_immune, color=['leiden'], legend_loc='on data', legend_fontsize=15,legend_fontoutline=0.5)

# To make the subtype comparable to those in the mouse datasets I am personalizing the colors used for the subtype variable
Human_immune.uns['subtype_colors'] = ['#fd9f4b','#5ba1d3', '#1fe98b']
sc.pl.umap(Human_immune, color=['subtype'], )
sc.pl.umap(Human_immune, color=['Original CellType'], legend_loc='on data', legend_fontsize=4,legend_fontoutline=0.5, add_outline=True)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))  # adjust grid size to your number of subtypes
axes = axes.flatten()

subtype = Human_immune.obs['subtype'].unique()

for ax, subtypes in zip(axes, subtype):
    sc.pl.umap(Human_immune[Human_immune.obs['subtype'] == subtypes].copy(),
               color='leiden', title=f"UMAP - {subtypes}", ax=ax, show=False)

plt.tight_layout()
plt.show()

In [ ]:
sc.pl.umap(Human_immune,color=['EGFR', 'PTPRZ1',
                               'TOP2A', 'MKI67',
                               'GFAP', 'AQP4',
                               'MOG', 'MBP',
                               'CSF1R', 'ITGAX',
                               'CLDN5','PDGFRB',
                               'CD3G', 'FOXJ1'],
           color_map='magma_r', size=5, ncols=3, use_raw=True)

In [ ]:

sc.tl.rank_genes_groups(Human_immune, groupby='leiden', method='wilcoxon', pts=True)
sc.pl.rank_genes_groups(Human_immune, sharey=False)

In [ ]:
markers = sc.get.rank_genes_groups_df(Human_immune, 
                                      group= None,
                                      pval_cutoff = 0.05)

columns = ['names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']
group_tables = []
group_labels = []  # will hold the group numbers for the top header row
unique_groups = markers['group'].unique()

for i, group in enumerate(unique_groups):
    group_df = markers[markers['group'] == group][columns].reset_index(drop=True)
    # Keep normal column names (no group prefix)
    group_tables.append(group_df)
    
    # Add spacer column except after the last group
    if i < len(unique_groups) - 1:
        spacer = pd.DataFrame(np.nan, index=group_df.index, columns=[""])
        group_tables.append(spacer)
    
    # For the top header row: group number only over 'names', blanks elsewhere
    group_labels.extend([group] + [""] * (len(columns) - 1))
    if i < len(unique_groups) - 1:
        group_labels.append("")  # spacer column has blank label

# Concatenate horizontally
final_table = pd.concat(group_tables, axis=1)

# Build MultiIndex for two header rows
final_table.columns = pd.MultiIndex.from_arrays([group_labels, final_table.columns])

# Save to CSV with multi-index headers
final_table.to_csv("Markers_Human_Clustering_leiden_second_round_111125.csv", index=False)
print("Saved: Markers_Human_Clustering_leiden_second_round_111125.csv")


In [ ]:
new_names = {'0':'Microglia',
             '1':'Tumor - OPC-like 1',
             '2':'Oligodendrocytes',
             '3':'Tumor - OPC-like 2',
             '4':'Tumor - OPC-like 3',
             '5':'Tumor - OPC-like 4',
             '6':'Tumor - Neuron-like',
             '7':'Macrophages',
             '8':'Tumor - Mesenchymal-like',
             '9':'Macrophages / Dendritic cells',
             '10':'Tumor - cycling',
             '11':'T Cells',
             '12':'Neurons',
             '13':'Oligodendrocytes',
             '14':'Tumor - OPC-like 5',
             '15':'Endothelial Cells',
             '16':'Tumor - Astro-like 1',
             '17':'Astrocytes 1',
             '18':'Macrophages',
             '19':'Tumor - Astro-like 2',
             '20':'Astrocytes 2',
             '21':'Tumor Astro-like 3',
             '22':'Pericytes',
             '23':'Doublets',
             '24':'Macrophages',
             '25':'Tumor - Astro-like 4'}

Human_immune.obs = Human_immune.obs.assign(CellType1 = Human_immune.obs.leiden.map(new_names))
sc.pl.umap(Human_immune,color=['CellType1'], legend_loc = 'on data', legend_fontsize=5, add_outline=True)

We need to filter it again and restart

In [ ]:
# filter doublet clusters
Human_immune_filter = Human_immune[~Human_immune.obs['CellType1'].isin(['Doublets'])]

In [ ]:
# restart the datasets using the backup merged version
mask = Human_backup.obs_names.isin(Human_immune_filter.obs_names)
Human_immune = Human_backup[mask].copy()

del mask

In [ ]:
#Keep the CellType identity and UMAP coordinates
Human_immune.obs['Original CellType']= Human_immune_filter.obs['Original CellType']
Human_immune.obs['CellType1']= Human_immune_filter.obs['CellType1']
Human_immune.obsm['Original_X_umap'] = Human_immune_filter.obsm['Original X_umap']
Human_immune.obsm['Filter1_X_umap'] = Human_immune_filter.obsm['X_umap']
del Human_immune.var

In [ ]:
#Reprocess the dataset
Human_immune.layers['counts'] = Human_immune.X.copy()

sc.pp.normalize_total(Human_immune, target_sum=1e4)
sc.pp.log1p(Human_immune)
Human_immune.layers['lognorm'] = Human_immune.X.copy()
Human_immune.raw = Human_immune

sc.pp.highly_variable_genes(Human_immune, min_mean=0.0125, max_mean=6, min_disp=0.10)
sc.pl.highly_variable_genes(Human_immune)

In [ ]:

sc.pp.regress_out(Human_immune, ['n_counts', 'percent_mito', 'percent_ribo', 'n_genes'], n_jobs=56)

In [ ]:
sc.pp.scale(Human_immune)
Human_immune.layers['scaled'] = Human_immune.X.copy()


sc.tl.pca(Human_immune, svd_solver='arpack', n_comps=100)
sc.pl.pca_variance_ratio(Human_immune, log=True, n_pcs = 100)

In [ ]:

sc.external.pp.harmony_integrate(Human_immune, key=['sample_label'],max_iter_harmony = 50)

In [ ]:
sc.pp.neighbors(Human_immune, n_neighbors=200, n_pcs=100,use_rep="X_pca_harmony")
sc.tl.umap(Human_immune, 
           #maxiter=100
          )
sc.tl.leiden(Human_immune, resolution=1.3)

In [ ]:

sc.pl.umap(Human_immune, color=['leiden'], legend_loc='on data', legend_fontsize=15,legend_fontoutline=0.5)

# To make the subtype comparable to those in the mouse datasets I am personalizing the colors used for the subtype variable
Human_immune.uns['subtype_colors'] = ['#fd9f4b','#5ba1d3', '#1fe98b']
sc.pl.umap(Human_immune, color=['subtype'], )
sc.pl.umap(Human_immune, color=['Original CellType'], legend_loc='on data', legend_fontsize=4,legend_fontoutline=0.5, add_outline=True)
sc.pl.umap(Human_immune, color=['CellType1'], legend_loc='on data', legend_fontsize=4,legend_fontoutline=0.5, add_outline=True)

In [ ]:

fig, axes = plt.subplots(3, 1, figsize=(6, 12))  # adjust grid size to your number of subtypes
axes = axes.flatten()

subtype = Human_immune.obs['subtype'].unique()

for ax, subtypes in zip(axes, subtype):
    sc.pl.umap(Human_immune[Human_immune.obs['subtype'] == subtypes].copy(),
               color='leiden',legend_loc='on data', legend_fontsize=15, legend_fontoutline=True, title=f"UMAP - {subtypes}", ax=ax, show=False)

plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(3, 1, figsize=(6, 12))  # adjust grid size to your number of subtypes
axes = axes.flatten()

subtype = Human_immune.obs['subtype'].unique()

for ax, subtypes in zip(axes, subtype):
    sc.pl.umap(Human_immune[Human_immune.obs['subtype'] == subtypes].copy(),
               color='leiden',legend_loc='on data', legend_fontsize=15, legend_fontoutline=True, title=f"UMAP - {subtypes}", ax=ax, show=False)

plt.tight_layout()
plt.show()

In [ ]:

sc.tl.rank_genes_groups(Human_immune, groupby='leiden', method='wilcoxon', pts=True)
sc.pl.rank_genes_groups(Human_immune, sharey=False)

In [ ]:
markers = sc.get.rank_genes_groups_df(Human_immune, 
                                      group= None,
                                      pval_cutoff = 0.05)

columns = ['names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']
group_tables = []
group_labels = []  # will hold the group numbers for the top header row
unique_groups = markers['group'].unique()

for i, group in enumerate(unique_groups):
    group_df = markers[markers['group'] == group][columns].reset_index(drop=True)
    # Keep normal column names (no group prefix)
    group_tables.append(group_df)
    
    # Add spacer column except after the last group
    if i < len(unique_groups) - 1:
        spacer = pd.DataFrame(np.nan, index=group_df.index, columns=[""])
        group_tables.append(spacer)
    
    # For the top header row: group number only over 'names', blanks elsewhere
    group_labels.extend([group] + [""] * (len(columns) - 1))
    if i < len(unique_groups) - 1:
        group_labels.append("")  # spacer column has blank label

# Concatenate horizontally
final_table = pd.concat(group_tables, axis=1)

# Build MultiIndex for two header rows
final_table.columns = pd.MultiIndex.from_arrays([group_labels, final_table.columns])

# Save to CSV with multi-index headers
final_table.to_csv("Markers_Human_Clustering_leiden_third_round_111125.csv", index=False)
print("Saved: Markers_Human_Clustering_leiden_third_round_111125.csv")

Looks like cluster 6 and 7 could be doublets, they seat between the microglia/macrophages and the tumors clusters and while cluster 6 seems to come from tumor background cluster 7 comes from a macrophage background, so it looks like it is a continuous.

In [ ]:
# filter doublet clusters
Human_immune_filter = Human_immune[~Human_immune.obs['leiden'].isin(['6', '7'])]

In [ ]:
# restart the datasets using the backup merged version
mask = Human_backup.obs_names.isin(Human_immune_filter.obs_names)
Human_immune = Human_backup[mask].copy()

del mask

In [ ]:
#Keep the CellType identity and UMAP coordinates
Human_immune.obs['Original CellType']= Human_immune_filter.obs['Original CellType']
Human_immune.obs['CellType1']= Human_immune_filter.obs['CellType1']
Human_immune.obsm['Original_X_umap'] = Human_immune_filter.obsm['Original_X_umap']
Human_immune.obsm['Filter1_X_umap'] = Human_immune_filter.obsm['Filter1_X_umap']
Human_immune.obsm['Filter2_X_umap'] = Human_immune_filter.obsm['X_umap']
del Human_immune.var

In [ ]:
#Reprocess the dataset
Human_immune.layers['counts'] = Human_immune.X.copy()

sc.pp.normalize_total(Human_immune, target_sum=1e4)
sc.pp.log1p(Human_immune)
Human_immune.layers['lognorm'] = Human_immune.X.copy()
Human_immune.raw = Human_immune

sc.pp.highly_variable_genes(Human_immune, min_mean=0.0125, max_mean=6, min_disp=0.10)
sc.pl.highly_variable_genes(Human_immune)

In [ ]:

sc.pp.regress_out(Human_immune, ['n_counts', 'percent_mito', 'percent_ribo', 'n_genes'], n_jobs=56)

In [ ]:
sc.pp.scale(Human_immune)
Human_immune.layers['scaled'] = Human_immune.X.copy()


sc.tl.pca(Human_immune, svd_solver='arpack', n_comps=100)
sc.pl.pca_variance_ratio(Human_immune, log=True, n_pcs = 100)

In [ ]:

sc.external.pp.harmony_integrate(Human_immune, key=['sample_label'],max_iter_harmony = 50)

In [ ]:
sc.pp.neighbors(Human_immune, n_neighbors=200, n_pcs=100,use_rep="X_pca_harmony")
sc.tl.umap(Human_immune, 
           #maxiter=100
          )
sc.tl.leiden(Human_immune, resolution=1.3)

In [ ]:

sc.pl.umap(Human_immune, color=['leiden'], legend_loc='on data', legend_fontsize=15,legend_fontoutline=0.5)

# To make the subtype comparable to those in the mouse datasets I am personalizing the colors used for the subtype variable
Human_immune.uns['subtype_colors'] = ['#fd9f4b','#5ba1d3', '#1fe98b']
sc.pl.umap(Human_immune, color=['subtype'], )
sc.pl.umap(Human_immune, color=['Original CellType'], legend_loc='on data', legend_fontsize=4,legend_fontoutline=0.5, add_outline=True)
sc.pl.umap(Human_immune, color=['CellType1'], legend_loc='on data', legend_fontsize=4,legend_fontoutline=0.5, add_outline=True)

In [ ]:

fig, axes = plt.subplots(3, 1, figsize=(6, 12))  # adjust grid size to your number of subtypes
axes = axes.flatten()

subtype = Human_immune.obs['subtype'].unique()

for ax, subtypes in zip(axes, subtype):
    sc.pl.umap(Human_immune[Human_immune.obs['subtype'] == subtypes].copy(),
               color='leiden',legend_loc='on data', legend_fontsize=15, legend_fontoutline=True, title=f"UMAP - {subtypes}", ax=ax, show=False)

plt.tight_layout()
plt.show()

In [ ]:
sc.pl.umap(Human_immune,color=['EGFR', 'PTPRZ1',
                               'TOP2A', 'MKI67',
                               'GFAP', 'AQP4',
                               'MOG', 'MBP',
                               'CSF1R', 'ITGAX',
                               'CLDN5','PDGFRB',
                               'CD3G', 'FOXJ1'], layer= 'lognorm',
           vmax='p99.5', vmin='p1',
           color_map='magma_r', size=5, ncols=3, use_raw=False)

In [ ]:
# Make a copy of the leiden labels
Human_immune.obs["leiden_split"] = Human_immune.obs["leiden"].astype(str)

# Mask for cluster 13
mask13 = Human_immune.obs["leiden_split"] == "13"

# Submask for Astrocytes 1 or 2
astro_mask = Human_immune.obs["CellType1"].isin(["Astrocytes 1", "Astrocytes 2"])

# Apply split
Human_immune.obs.loc[mask13 & astro_mask, "leiden_split"] = "13_Astro"
Human_immune.obs.loc[mask13 & ~astro_mask, "leiden_split"] = "13_other"
# Now you can use 'leiden_split' instead of 'leiden' for plotting/analysis
sc.pl.umap(Human_immune, color=["leiden_split"])

In [ ]:

sc.tl.rank_genes_groups(Human_immune, groupby='leiden_split', method='wilcoxon', pts=True)
sc.pl.rank_genes_groups(Human_immune, sharey=False)

In [ ]:
markers = sc.get.rank_genes_groups_df(Human_immune, 
                                      group= None,
                                      pval_cutoff = 0.05)

columns = ['names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']
group_tables = []
group_labels = []  # will hold the group numbers for the top header row
unique_groups = markers['group'].unique()

for i, group in enumerate(unique_groups):
    group_df = markers[markers['group'] == group][columns].reset_index(drop=True)
    # Keep normal column names (no group prefix)
    group_tables.append(group_df)
    
    # Add spacer column except after the last group
    if i < len(unique_groups) - 1:
        spacer = pd.DataFrame(np.nan, index=group_df.index, columns=[""])
        group_tables.append(spacer)
    
    # For the top header row: group number only over 'names', blanks elsewhere
    group_labels.extend([group] + [""] * (len(columns) - 1))
    if i < len(unique_groups) - 1:
        group_labels.append("")  # spacer column has blank label

# Concatenate horizontally
final_table = pd.concat(group_tables, axis=1)

# Build MultiIndex for two header rows
final_table.columns = pd.MultiIndex.from_arrays([group_labels, final_table.columns])

# Save to CSV with multi-index headers
final_table.to_csv("Markers_Human_Clustering_leiden_fourth_round_111125.csv", index=False)
print("Saved: Markers_Human_Clustering_leiden_fourth_round_111125.csv")

In [ ]:
marker_panel = {
    "Neuron-like": [
        "MAP2", "GRIA1", "GRIA2", "GRIA3", "SCN8A", "RYR3",
        "NRXN1", "NRXN3", "NLGN1", "NLGN4X", "CNTN1", "CHL1"
    ],
    "Astro-like": [
        "GFAP", "AQP4", "ALDH1L1", "SLC1A3", "GLUL", "SERPINA3"
    ],
    "Oligo-like": [
        "SOX10", "OLIG1", "OLIG2", "MBP", "MOG", "PLP1"
    ],
    "OPC-like": [
        "PDGFRA", "CSPG4", "SOX6", "SOX9", "NKX2-2"
    ],
    "Mesenchymal-like": [
        "CHI3L1", "COL8A1", "COL23A1", "SULF1", "LAMA2", "IGFBP7", "TAGLN"
    ],
    "Microglia": [
        "TMEM119", "P2RY12", "TREM2", "SALL1"
    ],
    "Macrophages": [
        "CD163", "MSR1", "SIGLEC1", "CCR2", "CD68"
    ],
    "Dendritic": [
        "ITGAX", "CD83", "HLA-DRA"
    ],
    "T cells": [
        "CD3D", "CD2", "IL7R"
    ]
}


In [ ]:
marker_panel_human = {
     "Peripheral Immune": [
        "CD3D", "CD2", "CD3E", "CD8A"
    ],
    "Peripheral Macrophages": [
        "MSR1", "SIGLEC1", "CD68"
    ],
    "Microglia": [
        "P2RY12","TMEM119", "TREM2", "SALL1"
    ],
    "Dendritic Cells": [
        "ITGAX", "CD83", "HLA-DRA"
    ],
    "Cycling": [
        "MKI67", "TOP2A", "CENPF"
    ],
    "Astrocytes / Tumor Astro-like": [
         "ALDH1L1",  "GLUL"
    ],
    "Tumor Mesenchymal-like": [
        "CHI3L1", "COL8A1", "IGFBP7"
    ],
    "Neurons / Tumor Neuron-like": [
        "MAP2",  "GRIA2", "GRIA3",  "NRXN1", 
    ],
    "Tumor OPC-like": [
       "SOX5", "PTPRG"
    ],
    "Endothelial Cells": [
        "CLDN5", "PECAM1", "CDH5"
    ],
    "Pericytes": [
        "PDGFRB", "RGS5"
    ],
    "Mature Oligodendrocytes": [
        "MBP", "MOG", "PLP1"
    ],
    
    "Reactive Astrocytes": [
       "GFAP", "AQP4"
    ],
    "Mature Neurons":[
        "GRIA1","SCN8A","NRXN3"
    ]
}


In [ ]:
sc.pl.dotplot(
    Human_immune,
    marker_panel_human,
    groupby="Final_CellType_simplified",
        standard_scale="var",
    dendrogram=True,
    #save="2025_12_10 - Human EGFR samples cell types dotplot.pdf"
)


To be sure of some cluster if they are tumor or not we can use copy number variations

In [ ]:
cnv.io.genomic_position_from_biomart(Human_immune, species='hsapiens',inplace=True,biomart_gene_id='hgnc_symbol')
cnv.tl.infercnv(
    Human_immune,
    reference_key="leiden_split",
    reference_cat=["0", "14",'2','4','9','12','16','19','18','11'], # non-tumor references
    layer = 'lognorm')
cnv.tl.cnv_score(Human_immune, groupby = 'leiden_split')


In [ ]:
#Subsample the addata object so the clusters are better represented on the heatmap
group_key = "leiden_split"
max_cells_per_cluster = 100  # adjust as needed

# Subsample each cluster
subsampled_indices = (
    Human_immune.obs
    .groupby(group_key)
    .apply(lambda x: x.sample(min(len(x), max_cells_per_cluster), random_state=42))
    .index.get_level_values(1)
)

# Create a new AnnData object with subsampled cells
adata_sub = Human_immune[subsampled_indices].copy()

# Build chromosome labels from chr_pos
chromosomes = list(adata_sub.uns['cnv']['chr_pos'].keys())
adata_sub.uns['cnv']['var_group_labels'] = chromosomes

In [ ]:
sns.set_context("talk", font_scale=0.7)  # "paper", "notebook", "talk", "poster"
cnv.pl.chromosome_heatmap(adata_sub,
                          groupby='leiden_split',
                          cmap="seismic",       # diverging colormap
                          figsize=(10, 12),     # compress vertically
                          dendrogram=True,     # optional: remove dendrogram for clarity
                          vmin=-0.15, vmax= 0.15, )

Let's name the clusters based on their DEG and CNV

In [ ]:
new_names = {'0':'Oligodendrocytes',
             '13_Astro':'Astrocytes',
             '18':'Astrocytes',
             '12':'Neurons',
             '16':'Endothelial Cells',
             '19':'Pericytes',
             '14':'Dendritic Cells',
             '2':'Microglia',
             '4':'Peripheral Macrophages',
             '9':'Peripheral Macrophages',
             '11':'Peripheral Immune',
             '1':'Tumor OPC-like 1',
             '3':'Tumor OPC-like 2',
             '5':'Tumor OPC-like 2',
             '7':'Tumor OPC-like 3',
             '20':'Tumor OPC-like 3',
             '13_other':'Tumor Astro-like',
             '15':'Tumor Astro-like',
             '17':'Tumor Astro-like',
             '21':'Tumor Astro-like',
             '8':'Tumor Neuron-like',
             '6':'Tumor Mesenchymal-like',
             '10':'Tumor Cycling'}

Human_immune.obs = Human_immune.obs.assign(Final_CellType = Human_immune.obs.leiden_split.map(new_names))
sc.pl.umap(Human_immune,color=['Final_CellType'], legend_loc = 'on data', legend_fontoutline=True,legend_fontsize=5, add_outline=True)

In [ ]:

#Save as pdf the final UMAPs with the defined cell identities
sc.pl.umap(Human_immune, color=['Final_CellType'],legend_fontsize = 7, 
           legend_loc='on data', legend_fontoutline=0.5, 
           add_outline=True,
           save=' - EGFRvIII Human samples cell types named 111925.pdf')

In [ ]:
for subtype in Human_immune.obs['subtype'].unique(): 
    save_file = " - EGFRvIII Human samples by subtypes - "+subtype+" 111925.pdf"
    sc.pl.umap(Human_immune, color='subtype', groups=[subtype], save=save_file, legend_loc=None)
    del save_file

Again we can analyze the genes markers for each cell type after merging together the corresponding leiden clusters.

In [ ]:
sc.tl.rank_genes_groups(Human_immune, groupby='Final_CellType_simplified')
sc.pl.rank_genes_groups(Human_immune, sharey=False)

In [ ]:
markers = sc.get.rank_genes_groups_df(Human_immune, 
                                      group= None,
                                      pval_cutoff = 0.05)

columns = ['names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']
group_tables = []
group_labels = []  # will hold the group numbers for the top header row
unique_groups = markers['group'].unique()

for i, group in enumerate(unique_groups):
    group_df = markers[markers['group'] == group][columns].reset_index(drop=True)
    # Keep normal column names (no group prefix)
    group_tables.append(group_df)
    
    # Add spacer column except after the last group
    if i < len(unique_groups) - 1:
        spacer = pd.DataFrame(np.nan, index=group_df.index, columns=[""])
        group_tables.append(spacer)
    
    # For the top header row: group number only over 'names', blanks elsewhere
    group_labels.extend([group] + [""] * (len(columns) - 1))
    if i < len(unique_groups) - 1:
        group_labels.append("")  # spacer column has blank label

# Concatenate horizontally
final_table = pd.concat(group_tables, axis=1)

# Build MultiIndex for two header rows
final_table.columns = pd.MultiIndex.from_arrays([group_labels, final_table.columns])

# Save to CSV with multi-index headers
final_table.to_csv("Markers Human Clustering named 121025.csv", index=False)
print("Saved: Markers Human Clustering named 121025.csv")

In [ ]:

Human_immune.obs['predicted_doublet'] = Human_immune.obs['predicted_doublet'].astype(bool)
Human_immune.write("/media/david/4TBNvMe/scRNAseq/Analyzed datasets/Human EgfrvIII filtered subtypes with clusters named 111925.h5ad")

del [new_names, subtype, markers2]

In [ ]:

Human_final =  Human_immune.copy()
Human_final.uns['Final_CellType_colors']=['#8c564b', #'Astrocytes'
                                          '#1f77b4', #'Dendrytic Cells'
                                          '#279e68', #'Endothelial'
                                          '#aa40fc', #'Microglia'
                                          '#d62728', #'Neurons'
                                          '#e377c2', #'Oligodendrocytes'
                                          '#b5bd61', #'Pericytes'
                                          '#aec7e8', #'Peripheral Immune'
                                          '#17becf', #'Peripheral Macrophages'
                                          '#ffbb78', #'Tumor Astro-like'
                                          '#98df8a', #'Tumor Cycling'
                                          '#ffbb95', #'Tumor Mesenchymal-like'
                                          '#ff9896', #'Tumor Neuron-like'
                                          '#c49c94', #'Tumor OPC-like 1'
                                          '#f7b6d2', #'Tumor OPC-like 2'
                                          '#9edae5', #'Tumor OPC-like 3'
                                         ]

matplotlib.rcParams['figure.figsize'] = (6.28, 4.46)
sc.settings.dpi = 600
sc.settings.dpi_save = 1200
ax = sc.pl.umap(
    Human_final,
    color=['Final_CellType'],
    add_outline=True,
    legend_loc='on data',
    legend_fontoutline=1.5,
    legend_fontsize=7,
    show=False  # don't immediately display
)

# Rasterize all scatter collections (the dots)
for coll in ax.collections:
    coll.set_rasterized(True)

# Save with desired DPI
ax.figure.savefig(
    "umap - Final Human datasets filtered and cleanup 111925.pdf",
    dpi=1200
)


In [ ]:
for subtype in Human_final.obs.subtype.unique(): 
    save_file = " - EGFRvIII Final Human samples by subtypes - "+subtype+" 111925.pdf"
    sc.pl.umap(Human_final, color='subtype', groups=[subtype], save=save_file, legend_loc=None)
    del save_file

save the final object

In [ ]:
Human_final.write("/media/david/4TBNvMe/scRNAseq/Analyzed datasets/Human EgfrvIII filtered subtypes with clusters named 111925.h5ad")
